# pythscribe — image filters **in the browser tab** (`@wasm` typed arrays, `js=` hook, no round-trip)

`browser_wasm_demos.ipynb` ran four *scalar* `@wasm` kernels client-side through Gradio's built-in
**`js=` event hook** (no custom component) — but its **Image filters** tab still ran on the **server**,
because the only in-tab image path was the list-buffer downscale component. This notebook closes that
gap (**v0.2.6 fix B**): the same `Array[uint8, 2]` image kernels (`threshold_lum`, `sobel`, and the M2c
`downscale_nn`) run **entirely in the tab** over first-class typed arrays, and the filtered image
updates **in the page** — the server does nothing per slider move (the input is a Gradio component value: fetched
from its `file=` URL once and cached in the tab; an upload reaches the server first, as with any `gr.Image`).

**How (still zero new components):** `demo.load(None, None, None, js=browser_image_loader_js({...}))`
imports the pythscribe FFI shim (`pythscribe/ffi/list_buffer.mjs`, from a `blob:` module URL) and a small
in-tab image client, and binds each kernel's **own `.wasm`** (served from its artifact dir, exactly as
`dispatch_image` does). A `js=` handler on the slider then does, per move:

```
gr.Image value (a file URL — fetched ONCE, cached in the tab)
  → createImageBitmap → canvas getImageData → H rows of Uint8Array(W*3)          (decode, in-tab)
  → ffi.call(kernel.wasm, fn, param_types, [img, out, h, w, …], {readBack: [out]})   (the WASM export)
  → `out` rows read back from WASM linear memory (the shim's self-checking write-back)
  → canvas → PNG data URL → the output gr.Image                                (display, in-tab)
```

No fallback on this path by design: the shim **refuses** (throws) rather than re-running on a JS twin,
so what ran is always the WASM export — and a failure is surfaced in the status line, never hidden.

Run in the **PythScribe (Gradio)** kernel (`pyths-gradio`). Requires the kernels' committed artifacts
(`python -m pythscribe.build examples/wasm-use-cases/kernels.py`).

## The app — drag the slider, switch the filter

Two tabs: **In-tab (client-side)** — the transform runs in your browser; and **Server round-trip** — the
*same* kernels wired through a Python `fn` (one round-trip per move; the server's counters advance).
The second tab is the measurement's **paired negative control**: driven through the same headless harness
it must *fail* the client-side assertions (requests > 0, counters > 0), so a silent fall-back to the server
could not pass the gate.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # examples/wasm-use-cases
from browser_image_filters_app import build_demo, KERNELS
from pythscribe import binding_of

for name, fn in KERNELS.items():
    b = binding_of(fn)
    print(f"{name:10s} -> {b.name:14s} artifact={b.artifact_status}  wasm={b.artifact.wasm.name if b.artifact else None}")

demo = build_demo()   # builds the js= loader: every kernel's contract + artifact is validated HERE
demo.launch(prevent_thread_lock=True, inline=True, quiet=True, server_name="127.0.0.1")

C:\Users\DELL\anaconda3\envs\pyths-gradio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


threshold  -> threshold_lum  artifact=resolved  wasm=threshold_lum.wasm
sobel      -> sobel          artifact=resolved  wasm=sobel.wasm
downscale  -> downscale_nn   artifact=resolved  wasm=downscale_nn.wasm


## Headless measurement — client-side, bit-for-bit, and the latency contrast

`browser_image_probe.py` (the same driver `tests/pythscribe/test_browser_image_filters_e2e.py` asserts
on) launches the app as a subprocess and drives it in a headless Chromium:

* **In-tab tab:** 10 threshold settings, then sobel, then nearest-neighbor downscale ×2 / ×3. It counts
  every request (and WebSocket) the page makes (`compute_requests` = anything under `/gradio_api/` that is not a static
  `file=` fetch) and reads the server-side kernel counters (`python_calls + server_calls`) before/after.
  For every output it captures BOTH the `out` rows read back from WASM memory AND the PNG actually
  displayed, and compares each with an independent NumPy reference **bit-for-bit** (plus the export's scalar return).
* **Server tab:** the same 10 threshold moves, then sobel and downscale, through the Python `fn` — the
  negative control (asserted **per kernel**) and the latency baseline.

(Runs as a subprocess: Windows' Proactor loop lets Playwright spawn Chromium; Jupyter's cannot.)

In [2]:
import json, subprocess, sys
p = subprocess.run([sys.executable, "browser_image_probe.py"], capture_output=True, text=True, cwd=str(Path.cwd()), timeout=900)
line = next((l for l in p.stdout.splitlines() if l.startswith("RESULT_JSON:")), None)
if line is None:
    print("probe failed:\n", (p.stderr or p.stdout)[-2000:])
else:
    r = json.loads(line[len("RESULT_JSON:"):])
    c, s = r["client"], r["server"]
    rows = [
        {"path": "in-tab (js= + typed-array @wasm)", "median slider->image ms": round(c["median_ms"], 1),
         "kernel ms (threshold)": round(c["kernel_median_ms"], 1), "compute requests": len(c["compute_requests"]),
         "server kernel calls": c["counters_delta"], "server work per slider move": "none"},
        {"path": "server round-trip (same kernels)", "median slider->image ms": round(s["median_ms"], 1),
         "kernel ms (threshold)": None, "compute requests": len(s["compute_requests"]),
         "server kernel calls": s["counters_delta"], "server work per slider move": "one round-trip + the kernel"},
    ]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows).set_index("path"))
        fid = pd.DataFrame([{"output": k, **v} for k, v in r["fidelity"].items()]).set_index("output")
        display(fid)
    except ImportError:
        for row in rows: print(row)
        for k, v in r["fidelity"].items(): print(k, v)
    print(f"image {r['image']}; sobel kernel {c['sobel_kernel_ms']:.1f} ms in-tab; .wasm fetched {len(c['wasm_requests'])}x (once per kernel), "
          f"static file fetches {len(c['file_requests'])}; layout {c['layout']}")
    print("in-tab status:", c["status"])
    print("server status:", s["status"])
    assert c["compute_requests"] == [] and c["counters_delta"] == 0, "the in-tab path touched the server"
    assert len(s["compute_requests"]) > 0 and all(v > 0 for v in s["counters_delta_per_kernel"].values()), "the negative control did not trip"
    assert r["all_bit_for_bit"], "an in-tab output differs from the NumPy reference"
    print(f"ALL {len(r['fidelity'])} outputs bit-for-bit == NumPy (rows read back from WASM memory AND the displayed PNG); "
          f"in-tab {c['median_ms']:.1f} ms vs server {s['median_ms']:.1f} ms")

,median slider->image ms,kernel ms (threshold),compute requests,server kernel calls,server work per slider move
path,,,,,
in-tab (js= + typed-array @wasm),18.7,7.0,0,0,none
server round-trip (same kernels),126.7,NaN,28,14,one round-trip + the kernel


,shape,rows_equal,rows_mismatches,shown_equal,ret,ret_equal
output,,,,,,
threshold@90,"[480, 1920]",True,0,True,307200,True
threshold@160,"[480, 1920]",True,0,True,307200,True
threshold@60,"[480, 1920]",True,0,True,307200,True
threshold@200,"[480, 1920]",True,0,True,307200,True
threshold@120,"[480, 1920]",True,0,True,307200,True
threshold@30,"[480, 1920]",True,0,True,307200,True
threshold@180,"[480, 1920]",True,0,True,307200,True
threshold@150,"[480, 1920]",True,0,True,307200,True
threshold@70,"[480, 1920]",True,0,True,307200,True


image [480, 640, 3]; sobel kernel 41.1 ms in-tab; .wasm fetched 3x (once per kernel), static file fetches 4; layout pyths-0.2.5-array-v2
in-tab status: downscale: 0.7 ms @wasm in-tab (5.4 ms incl. decode/encode), 213x160, 0 round-trips, layout pyths-0.2.5-array-v2
server status: threshold: 8.4 ms on the SERVER (mode=server); 1 round-trip; python_calls+=0 server_calls+=1
ALL 13 outputs bit-for-bit == NumPy (rows read back from WASM memory AND the displayed PNG); in-tab 18.7 ms vs server 126.7 ms


## Reading the numbers

* **Client-side, verified two ways:** 0 compute requests from the page during the interaction (only the
  input PNG once and each kernel's `.wasm` once, both static), and a 0 delta on every kernel's server-side
  counter — while the server tab, through the same harness, produces both for every kernel (the control trips).
  Not claimed: that the image never reaches the server — it is a `gr.Image` value (served / uploaded by Gradio).
* **Bit-for-bit:** every in-tab output equals the NumPy reference — both the `out` typed array the shim
  read back from WASM linear memory and the PNG the page actually displays, plus the export's scalar
  return — for 13 (kernel, setting) pairs, including the `oh/ow`-shaped downscale (a 2-D `Array[uint8, 2]`
  out-buffer of a different shape). A browserless randomized differential (`test_browser_image_loader.py`)
  runs the same `.wasm` on the server path against the same references on random images.
* **Latency:** the in-tab median is the decode-cached → kernel → encode → display path; the server median
  is the same kernel plus the upload/round-trip. Contrast this with `browser_wasm_demos.ipynb`, whose
  image tab could only offer the server path.

**Reuse** (any `(img: Array[uint8, 2], out: Array[uint8, 2], h: int, w: int, *extras) -> int` kernel,
or one with its own `oh, ow` output dims; extras are passed BY NAME):

```python
from pythscribe.gradio.browser import browser_image_loader_js
demo.load(None, None, None, js=browser_image_loader_js({"threshold": threshold_lum, "sobel": sobel}))
slider.change(None, [image, kind, slider], [out_image, status],
              js="(img, kind, thr) => window.pythscribeImage.filter(img, kind, kind === 'threshold' ? {thr: thr * 3} : {})")
```

A stub is published before the client finishes loading, so an early slider move gets a 'loading' status (and a
load failure — e.g. a CSP blocking `import(blob:)` — is reported in the status line, never swallowed).

The gate: `tests/pythscribe/test_browser_image_filters_e2e.py` (client-side + per-kernel negative control,
fidelity + negative control, latency) and `test_browser_image_loader.py` (the contract's refusals + the
randomized differential).